# AIDP TPC-DS Benchmark — V1.2 Power Test (Portable)

Runs the official TPC-DS v4.0.0 benchmark on AIDP in **any** OCI tenancy — no hardcoded namespace, no private bucket fetch.

**Prerequisites (one-time setup):**
- AIDP Workbench cluster, Python 3.10+, Spark 3.5.x
- Cluster has internet egress (Cell 2 installs the wheel from the GitHub Release)
- IAM policy on the cluster's dynamic group — see `REQUIREMENTS.md`
- AIDP Unified Catalog is the default catalog for the Spark session

**To run:** set `SF` in Cell 1, then **Run All**. Total customer code: 1 line.

Version: `aidp_benchmark >= 0.1.2`

## Cell 1 — Configuration

The only knob a customer touches. `SF` is the TPC-DS scale factor (dataset size in GB). Start with `1` to validate end-to-end, then move to `10` or `100`.

In [ ]:
# CELL 1 — CONFIGURATION: change only this
SF = 10   # Scale factor: 1 | 10 | 100
print(f"Configured for SF={SF}")

## Cell 2 — Install `aidp-benchmark`

Downloads the wheel from an OCI bucket via Hadoop FS (Resource Principal — no credentials needed), then installs it. Set `BUCKET` to the bucket where you uploaded the wheel.

> **Note:** After the PR merges and a PyPI release is available, this cell will be replaced with `%pip install aidp-benchmark`.

In [ ]:
import subprocess, sys, os, shutil, re, glob
from py4j.java_gateway import java_import

# ── SET THIS to the bucket where you uploaded aidp_benchmark-0.1.2-py3-none-any.whl ──
BUCKET = "aidp-tpcds-dev"
# ─────────────────────────────────────────────────────────────────────────────────────

TARGET   = "/tmp/aidp_pkg"
WHL_NAME = "aidp_benchmark-0.1.2-py3-none-any.whl"
DL_DIR   = "/tmp/aidp_dl"
DL_PATH  = f"{DL_DIR}/{WHL_NAME}"

# 1. Clean previous installs and kernel cache
if os.path.isdir(TARGET): shutil.rmtree(TARGET)
if os.path.isdir(DL_DIR):  shutil.rmtree(DL_DIR)
os.makedirs(TARGET, exist_ok=True)
os.makedirs(DL_DIR,  exist_ok=True)
os.chmod(DL_DIR, 0o777)
for k in list(sys.modules.keys()):
    if "aidp_benchmark" in k or "duckdb" in k:
        del sys.modules[k]

# 2. Auto-detect storage namespace from catalog
ns = None
for row in spark.sql("SHOW DATABASES").collect()[:10]:
    try:
        for r in spark.sql(f"DESCRIBE DATABASE EXTENDED `{row[0]}`").collect():
            m = re.search(r'oci://[^@]+@([^/]+)/', str(r))
            if m:
                ns = m.group(1)
                break
    except Exception:
        continue
    if ns:
        break
if not ns:
    raise RuntimeError("Cannot auto-detect OCI namespace — ensure catalog has at least one database with an OCI location.")
print(f"Namespace detected: {ns}")

# 3. Download wheel from OCI bucket via Hadoop FS
java_import(spark._jvm, 'org.apache.hadoop.fs.FileSystem')
java_import(spark._jvm, 'org.apache.hadoop.fs.Path')
fs = spark._jvm.FileSystem.get(
    spark._jvm.java.net.URI.create(f"oci://{BUCKET}@{ns}/"),
    spark.sparkContext._jsc.hadoopConfiguration()
)
fs.copyToLocalFile(False,
    spark._jvm.Path(f"oci://{BUCKET}@{ns}/{WHL_NAME}"),
    spark._jvm.Path(DL_PATH))
print(f"Downloaded {WHL_NAME}")

# 4. Install aidp_benchmark
r = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     DL_PATH, "--target", TARGET, "--no-deps", "-q"],
    capture_output=True, text=True, cwd="/tmp")
if r.returncode != 0:
    raise RuntimeError(r.stderr[-500:])
print("Installed aidp_benchmark")

# 5. Install duckdb 1.0.0 (pinned — newer versions break with --target on AIDP)
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "duckdb==0.10.3",
     "--target", TARGET, "-q",
     "--index-url", "https://pypi.org/simple/",
     "--trusted-host", "pypi.org",
     "--trusted-host", "files.pythonhosted.org"],
    capture_output=True, text=True, cwd="/tmp")
if r.returncode != 0:
    raise RuntimeError(r.stderr[-500:])

# 6. Move _duckdb native extension to top of TARGET (required for correct import resolution)
for f in glob.glob(f"{TARGET}/duckdb/_duckdb*.so"):
    shutil.copy(f, TARGET)
print("Installed duckdb")

# 7. Prepend to sys.path
sys.path.insert(0, TARGET)
import aidp_benchmark, duckdb
print(f"aidp_benchmark={aidp_benchmark.__version__}  duckdb={duckdb.__version__}  — ready")

## Cell 3 — Generate data + run the benchmark

Three things happen here, automatically:

1. **Auto-detect tenancy** — namespace, region, and compartment come from the cluster's Resource Principal. No hardcoded tenancy strings.
2. **Generate + upload data** — DuckDB `dsdgen` produces 24 Parquet files locally, then streams them to `oci://tpcds-benchmark-sf{SF}/`. Idempotent: re-runs skip tables already in the bucket.
3. **Run all 103 queries** — TPC-DS v4.0.0 with the official a/b splits. Sequential power test. GeoMean is the headline metric.

Expect ~10s GeoMean at `SF=10` on a default AIDP cluster (V1.1 baseline: 9.67s).

In [ ]:
# CELL 3 — Run the benchmark (namespace auto-detected via Resource Principal)
from aidp_benchmark import TPCDSDataGenerator, AIDPBenchmark

gen = TPCDSDataGenerator(scale_factor=SF)
gen_result = gen.generate(spark=spark)
print(f"Data ready: bucket={gen_result.bucket_name} "
      f"status={gen_result.bucket_status} "
      f"duration={gen_result.duration_seconds}s")
print(f"Upload: {gen_result.upload_summary}")

result = AIDPBenchmark(gen).run_power_test()
print(result.summary())

## Cell 4 — Save results to AIDP Unified Catalog

Writes two Iceberg tables in the `aidp_benchmark` catalog database:
- `aidp_benchmark.run_summary` — one row per benchmark run (GeoMean, P50/P95/P99, queries passed/failed)
- `aidp_benchmark.query_results` — one row per query per run (elapsed time, status, execution plan)

Append-only. Safe to re-run.

In [ ]:
# CELL 4 — Save to AIDP Unified Catalog (Iceberg)
result.save()
print(f"Saved run {result.run_id}")
print("  → aidp_benchmark.run_summary")
print("  → aidp_benchmark.query_results")

## Cell 5 — Verify results

Reads the run back out of the catalog as a sanity check. Shows the summary row + top 10 slowest queries.

In [ ]:
# CELL 5 — Verify: read back from catalog
print("=== run_summary ===")
spark.sql(f"""
    SELECT run_id, scale_factor, queries_passed, queries_failed,
           geomean_seconds, p95_seconds, end_time
    FROM aidp_benchmark.run_summary
    WHERE run_id = '{result.run_id}'
""").show(truncate=False)

print("=== top 10 slowest queries ===")
spark.sql(f"""
    SELECT query_name, elapsed_seconds, status
    FROM aidp_benchmark.query_results
    WHERE run_id = '{result.run_id}'
    ORDER BY elapsed_seconds DESC LIMIT 10
""").show(truncate=False)

## Cell 6 — Optional one-time cleanup (DISABLED BY DEFAULT)

Drops the entire `aidp_benchmark` catalog database. **Destructive — wipes ALL prior runs from `run_summary` AND `query_results`.** Uncomment only if you want a clean slate.

In [ ]:
# CELL 6 (OPTIONAL, ONE-TIME) — Wipe the aidp_benchmark catalog
# WARNING: deletes ALL prior benchmark runs. Uncomment ONLY for first-run cleanup.
#
# spark.sql("DROP DATABASE IF EXISTS aidp_benchmark CASCADE")
# print("Dropped — next run will recreate the database from scratch.")
print("Cell 6 skipped (wipe commented out — uncomment manually to use).")

---
**Done.** Compare results with `SELECT * FROM aidp_benchmark.run_summary ORDER BY end_time DESC LIMIT 5`. File issues at the project's GitHub repo.

> **Note:** Results produced by this toolkit are not comparable to official TPC Benchmark Results.